# Minimal Calculator Agent Demo

This notebook shows the simplest framework path for a small agent:

- use `PlannerNode` directly
- use `ResponseNode` directly
- provide prompts on the nodes themselves
- build a small LangGraph

No subclassing is needed here because both nodes now have basic built-in behavior.


In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook path.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

required_packages = ["langgraph", "transformers", "accelerate", "sentencepiece"]
for package in required_packages:
    try:
        __import__(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print(f"Using repository root: {repo_root}")
print(f"Python executable: {sys.executable}")


Using repository root: /Users/saketm10/Projects/openclaw_agents
Python executable: /Users/saketm10/Projects/openclaw_agents/.venv/bin/python3.11


In [2]:
from langgraph.graph import END, START, StateGraph

from src.agents.nodes import AgentState, PlannerNode, ResponseNode
from src.llm.qwen import Qwen3_1_7BLLM


qwen_llm = Qwen3_1_7BLLM(model_name="Qwen/Qwen3-1.7B", max_new_tokens=128, enable_thinking=False)

planner_node = PlannerNode(
    llm=qwen_llm,
    system_prompt=(
        "You are a calculator assistant. "
        "Solve the arithmetic problem and return only the final answer in one short line."
    ),
    user_prompt="Question: {user_input}",
)
response_node = ResponseNode(
    llm=qwen_llm,
    system_prompt=(
        "You are a calculator assistant. "
        "Turn the computed answer into a short user-facing reply in English language"
    ),
    user_prompt="User question: {user_input}\nComputed answer: {observation}",
    default_response="I could not compute that.",
)

graph = StateGraph(AgentState)
graph.add_node("plan", planner_node.execute)
graph.add_node("respond", response_node.execute)
graph.add_edge(START, "plan")
graph.add_conditional_edges(
    "plan",
    planner_node.route,
    {"respond": "respond", "end": END, "act": "respond"},
)
graph.add_edge("respond", END)
compiled_graph = graph.compile()


In [3]:
state = compiled_graph.invoke(
    {
        "user_input": "What is 12 * (3 + 4)?",
        "steps": 0,
    }
)
state["response"]


'The result of 12 * (3 + 4) is 84.'